## Optional standalone cell: download the full SynthHuman release to Drive
Run this cell only when you intentionally want the entire dataset. It is disabled by default, downloads chunks sequentially, stores extracted data and resume markers on Drive, and keeps an interrupted ZIP to support continuation. Allow about 330 GB for the release plus extra space for extracted files. The full download is not needed for the 500-sample pilot.

In [ ]:
# Optional full SynthHuman download — standalone from the training pipeline.
# Keep False for normal notebook runs. Set True only after confirming Drive has enough free space.
RUN_FULL_DOWNLOAD = False
if not RUN_FULL_DOWNLOAD:
    print('Full SynthHuman download skipped. Set RUN_FULL_DOWNLOAD=True in this cell only when you intend to download the full ~330 GB release.')
else:
    import subprocess, sys, zipfile, json
    from pathlib import Path
    if not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/AppleCapstoneRelighting')
    SYNTHHUMAN_PARENT = DRIVE_ROOT / 'dataset' / 'sources'
    SYNTHHUMAN_ROOT = SYNTHHUMAN_PARENT / 'SynthHuman'
    DOWNLOAD_STATE = SYNTHHUMAN_PARENT / '.synthhuman_download_state'
    DOWNLOAD_CACHE = SYNTHHUMAN_PARENT / '.synthhuman_zip_cache'
    DAVID_REPO = Path('/content/DAViD')
    for folder in (SYNTHHUMAN_ROOT, DOWNLOAD_STATE, DOWNLOAD_CACHE): folder.mkdir(parents=True,exist_ok=True)
    if not (DAVID_REPO / 'download_data.py').is_file():
        subprocess.run(['git','clone','--depth','1','https://github.com/microsoft/DAViD.git',str(DAVID_REPO)],check=True)
    TOTAL_SAMPLES, CHUNK_SIZE = 298008, 5000
    BASE_URL = 'https://facesyntheticspubwedata.z6.web.core.windows.net/iccv-2025/data'
    for chunk_idx in range((TOTAL_SAMPLES+CHUNK_SIZE-1)//CHUNK_SIZE):
        first_id=chunk_idx*CHUNK_SIZE
        sample_count=min(CHUNK_SIZE,TOTAL_SAMPLES-first_id)
        last_id=first_id+sample_count-1
        marker=DOWNLOAD_STATE/f'SynthHuman_{chunk_idx:04d}.complete'
        endpoint_names=[f'{kind}_{sample_id:07d}.{ext}' for sample_id in (first_id,last_id) for kind,ext in (('rgb','png'),('alpha','png'),('depth','exr'),('normal','exr'))]
        if all((SYNTHHUMAN_ROOT/name).is_file() for name in endpoint_names):
            if not marker.exists(): marker.write_text(json.dumps({'chunk':chunk_idx,'status':'already present before resumable downloader'}))
            print(f'Chunk {chunk_idx+1:02d}/60 already present; skipping.')
            continue
        zip_path=DOWNLOAD_CACHE/f'SynthHuman_{chunk_idx:04d}.zip'
        url=f'{BASE_URL}/SynthHuman_{chunk_idx:04d}.zip'
        print(f'Downloading/extracting chunk {chunk_idx+1:02d}/60 ({sample_count} samples) directly under Drive...')
        subprocess.run(['wget','--continue','--no-check-certificate','--secure-protocol=TLSv1_2','-O',str(zip_path),url],check=True)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(SYNTHHUMAN_ROOT)  # ZIP CRC is checked while members are extracted.
        missing=[name for name in endpoint_names if not (SYNTHHUMAN_ROOT/name).is_file()]
        if missing: raise RuntimeError(f'Chunk {chunk_idx:04d} extraction is incomplete; ZIP retained for retry: {missing}')
        marker.write_text(json.dumps({'chunk':chunk_idx,'first_sample_id':first_id,'last_sample_id':last_id,'sample_count':sample_count,'url':url},indent=2))
        zip_path.unlink(missing_ok=True)
        print(f'Chunk {chunk_idx+1:02d}/60 complete; verified endpoints and removed ZIP.')
    print('Full SynthHuman download pass finished. Dataset:',SYNTHHUMAN_ROOT)
    print('RGB sample files:',len(list(SYNTHHUMAN_ROOT.glob('rgb_*.png'))))